# Pipeline CNN (PyTorch) — Imagens Intraorais Odontológicas

Pipeline:
1. Preparar o dataset (por sujeito) em um layout `train/val/test` por classe, compatível com `ImageFolder`
2. Construir os `DataLoader`s de treino, validação e teste
3. Treinar uma **CNN simples**, implementada do zero (sem pré-treino / transfer learning)
4. Acompanhar loss e acurácia por época
5. Avaliar no conjunto de teste e visualizar a matriz de confusão
6. Classificar imagens individuais e salvar o modelo treinado

Este notebook reaproveita as classes e funções de `src/pytorch_classifier`, para que o comportamento seja idêntico ao da CLI (`torch-train` / `torch-predict`).

## 1. Importações

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import random

import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from src.pytorch_classifier.dataset import build_dataloaders, resolve_imagefolder_root
from src.pytorch_classifier.model import DentalCNN, ModelConfig
from src.pytorch_classifier.predict import predict
from src.pytorch_classifier.trainer import Trainer
from src.pytorch_classifier.utils import get_device, save_checkpoint, set_seed

%matplotlib inline
print('Importações OK')

## 2. Configuração dos caminhos

In [ ]:
DATASET_ROOT = PROJECT_ROOT / "data" / "dataset"
MODEL_OUT    = PROJECT_ROOT / "artifacts" / "torch_model.pth"

IMAGE_SIZE    = 128
GRAYSCALE     = True
BATCH_SIZE    = 32
EPOCHS        = 20
PATIENCE      = 5      # early stopping: nº de épocas sem melhora na val loss antes de parar
MIN_DELTA     = 0.0    # melhora mínima na val loss para zerar a paciência
LEARNING_RATE = 1e-3
SEED          = 42

set_seed(SEED)
device = get_device()
print(f"Dispositivo  : {device}")
print(f"Dataset      : {DATASET_ROOT}")
print(f"Modelo (out) : {MODEL_OUT}")

## 3. Carregamento do Dataset

In [ ]:
# Materializa (uma única vez) o dataset por sujeito em train/val/test por classe.
# A divisão é feita por sujeito, para não vazar dados do mesmo paciente entre conjuntos.
imagefolder_root = resolve_imagefolder_root(DATASET_ROOT, seed=SEED)
print(f"\nLayout ImageFolder em: {imagefolder_root}")

## 4. Visualização de algumas imagens

In [ ]:
classes_preview = sorted(p.name for p in (imagefolder_root / "train").iterdir() if p.is_dir())

fig, axes = plt.subplots(1, len(classes_preview), figsize=(15, 3))
for ax, classe in zip(axes, classes_preview):
    exemplo = random.choice(list((imagefolder_root / "train" / classe).glob("*")))
    ax.imshow(Image.open(exemplo))
    ax.set_title(classe, fontsize=9)
    ax.axis('off')
fig.suptitle("Um exemplo aleatório por classe (conjunto de treino)", y=1.02)
plt.tight_layout()
plt.show()

## 5. Criação do DataLoader

In [ ]:
train_loader, val_loader, test_loader, classes = build_dataloaders(
    imagefolder_root,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    grayscale=GRAYSCALE,
)
NUM_CLASSES = len(classes)

print(f"Classes ({NUM_CLASSES}): {classes}")
print(f"Batches — treino: {len(train_loader)}  val: {len(val_loader)}  teste: {len(test_loader)}")

## 6. Construção da CNN

In [ ]:
config = ModelConfig(image_size=IMAGE_SIZE, grayscale=GRAYSCALE)
model = DentalCNN(num_classes=NUM_CLASSES, in_channels=config.in_channels)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {total_params:,}")
model

## 7. Configuração do treinamento

In [ ]:
trainer = Trainer(model, device, learning_rate=LEARNING_RATE)
print(f"Otimizador : {trainer.optimizer.__class__.__name__} (lr={LEARNING_RATE})")
print(f"Loss       : {trainer.criterion.__class__.__name__}")

## 8. Loop de treinamento

In [ ]:
history = trainer.fit(train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA)

## 9. Gráfico da Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.train_loss, label="Treino")
plt.plot(history.val_loss, label="Validação")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Curva de Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Gráfico da Accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.train_accuracy, label="Treino")
plt.plot(history.val_accuracy, label="Validação")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.title("Curva de Acurácia")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Avaliação final

In [ ]:
test_loss, test_acc = trainer.evaluate(test_loader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

## 12. Matriz de confusão

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        y_pred.extend(outputs.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

print(classification_report(y_true, y_pred, target_names=classes))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classes, cmap="Blues", xticks_rotation=45, ax=ax,
)
ax.set_title(f"Matriz de Confusão — Teste (acurácia = {test_acc:.4f})")
plt.tight_layout()
plt.show()

## 13. Predição em imagens individuais

In [ ]:
classe_exemplo = random.choice(classes)
exemplo_path = random.choice(list((imagefolder_root / "test" / classe_exemplo).glob("*")))
pred = predict(exemplo_path, model, classes, config, device)

plt.figure(figsize=(3, 3))
plt.imshow(Image.open(exemplo_path))
plt.title(f"Previsto: {pred.label}")
plt.axis('off')
plt.show()

print(f"Imagem          : {exemplo_path.name}")
print(f"Classe real     : {classe_exemplo}")
print(f"Classe prevista : {pred.label}")
print("Probabilidades  :")
for cls, prob in sorted(pred.probabilities.items(), key=lambda kv: -kv[1]):
    print(f"  {cls:<20} {prob:.1%}")

## 14. Salvamento do modelo

In [ ]:
save_checkpoint(MODEL_OUT, model, classes, config)
print(f"Modelo salvo em: {MODEL_OUT}")